# Causal models as Python functions
Run these cells in order. The named V assignments are the intervention boundaries.


In [ ]:
from causalab.causal.model import CausalModel
from causalab.causal import Dom, FamilyDom, V, family, mechanism, submodel

@submodel
def compare(a, b):
    equal = V(a == b)
    return equal

@mechanism
def equations(values: FamilyDom(Dom(range(3)), size=3)):
    left = compare(values[0], values[1])
    right = compare(values[1], values[2])
    result = V(left == right)
    raw_input = V(str(values), domain=Dom(str))  # noqa: F841
    raw_output = V(str(result), domain=Dom(str))  # noqa: F841
    return result

model = CausalModel(equations)
model.parents


In [ ]:
trace = model.new_trace({"values": [0, 0, 1]})
assert trace["result"] is False
trace["left.equal"] = False
assert trace["result"] is True
trace.to_dict()


In [ ]:
bound = 3
@mechanism
def countdown(initial: Dom(range(bound + 1))):
    state = family(size=bound + 1)
    state[0] = initial
    for t in range(bound):
        state[t + 1] = max(0, state[t] - 1)
    raw_input = V(str(initial), domain=Dom(str))  # noqa: F841
    raw_output = V(str(state[bound]), domain=Dom(str))  # noqa: F841
    return state[bound]

steps = CausalModel(countdown)
assert steps.new_trace({"initial": 3})["state[3]"] == 0
steps.parents
